# Chapter 2 Lab — Tokenization: BPE vs Whitespace, and OOV Handling

**Estimated time:** 15 minutes · **Read-and-run.** Pure-Python mini-BPE (no installs).
An optional final cell uses `tiktoken` if you have it.

In [ ]:
import re, collections
corpus = ['low lower lowest', 'newer wider', 'new news newest', 'the network learns']

def get_vocab(corpus):
    v = collections.Counter()
    for line in corpus:
        for w in line.split():
            v[' '.join(list(w)) + ' </w>'] += 1
    return v

def merges_step(vocab):
    pairs = collections.Counter()
    for word, freq in vocab.items():
        sym = word.split()
        for i in range(len(sym)-1):
            pairs[(sym[i], sym[i+1])] += freq
    if not pairs: return None, vocab
    best = max(pairs, key=pairs.get)
    bigram = re.escape(' '.join(best)); patt = re.compile(r'(?<!\S)'+bigram+r'(?!\S)')
    vocab = {patt.sub(''.join(best), w): f for w,f in vocab.items()}
    return best, vocab

In [ ]:
vocab = get_vocab(corpus)
merges = []
for _ in range(12):
    best, vocab = merges_step(vocab)
    if best is None: break
    merges.append(best)
print('learned merges:'); [print('  ', a, '+', b) for a,b in merges]

## Encode a word, including an out-of-domain one

In [ ]:
def encode(word, merges):
    sym = list(word) + ['</w>']
    for a,b in merges:
        i=0; merged=[]
        while i < len(sym):
            if i < len(sym)-1 and sym[i]==a and sym[i+1]==b:
                merged.append(a+b); i+=2
            else: merged.append(sym[i]); i+=1
        sym = merged
    return sym

for w in ['newer', 'lowest', 'transformer']:  # 'transformer' is OOD here
    toks = encode(w, merges)
    print(f'{w:12s} -> {len(toks)} tokens  {toks}')
print('\nNote: the OOD word falls back to many short/character tokens — it is never <UNK>.')

In [ ]:
# OPTIONAL: real subword tokenizer, only if installed.
try:
    import tiktoken
    enc = tiktoken.get_encoding('cl100k_base')
    for s in ['transformer', 'antidisestablishmentarianism', 'GPU kernels']:
        print(f'{s:35s} -> {len(enc.encode(s))} tokens')
except Exception as e:
    print('tiktoken not installed (optional):', type(e).__name__)

## Key takeaway
Subword BPE keeps a bounded vocabulary while still representing unseen words (as more,
smaller tokens) instead of emitting `<UNK>`. Token count — not word count — drives context
usage and cost, which the exam expects you to reason about quantitatively.